# Generación del Dataset Sintético AndinaSense

- **Asignatura:** Inteligencia Artificial
- **Grupo:** 06
- **Integrantes:** Bautista De la Cruz Claudia Daniela, Carrascal Castro Priscila Maria, Ccahuana Quiñones Judith Valeria, Gil Sixi Alberto Luis, Medrano Ayma Nikol Arlet, Rosales Trinidad Jeanmarco Miguel, Saire Tello Fernando José

**Objetivo de este archivo:** generar de forma reproducible el dataset sintético `andinasense_parcelas.csv` que comparten los Informes 3 a 6. Este notebook se ejecuta una sola vez para producir el CSV; los notebooks de análisis únicamente lo cargan.

## Descripción de la generación

El dataset es sintético y propio del grupo, generado con numpy y pandas. Es reproducible con la semilla SEED = 42.

Volumen: 1500 registros (filas) y 13 variables (columnas), con mezcla de numéricas y categóricas.

Variables:

- id_parcela: identificador de texto (PARC-00001 en adelante).
- region: categórica (La Libertad 50%, Lambayeque 30%, Ica 20%).
- variedad: categórica (Biloxi, Ventura, Emerald, Rocio).
- superficie_ha: continua uniforme entre 2 y 12.
- temp_promedio: temperatura en °C, rango 14 a 26.
- humedad_suelo: humedad en %, rango 20 a 95.
- ph_suelo: pH del suelo, rango 4.2 a 8.0.
- horas_sol: horas de sol, rango 5 a 12.
- riego_mm: lámina de riego en mm, rango 200 a 900.
- fertilizante_kg: fertilizante en kg, rango 18 a 617.
- densidad_plantas: plantas por hectárea, rango 2000 a 6000.
- rendimiento_kg_ha: variable objetivo continua (para regresión).
- calidad_cosecha: variable objetivo categórica Baja, Media, Alta (para clasificación).

Reglas y correlaciones introducidas:

- Las parcelas provienen de 4 zonas agroecológicas latentes con proporciones controladas (Óptima 35%, Estrés Hídrico 25%, Desbalance de pH 20%, Multi-estrés Leve 20%). Cada zona tiene un perfil de medias por variable de suelo, clima y manejo. La zona se usa solo para generar y no se guarda como columna.
- La dispersión de cada zona está calibrada para un solapamiento realista (correlaciones y ruido, no grupos perfectos).

Generación de las variables objetivo:

- rendimiento_kg_ha se deriva de una función de respuesta agronómica con penalización cuadrática (óptimo de temperatura 20 °C y de pH 5.5), efectos de humedad y sol, y ruido gaussiano.
- calidad_cosecha se obtiene por terciles del rendimiento (Baja, Media, Alta).

Estructura para el preprocesamiento: se introducen valores atípicos y nulos intencionales en columnas de sensores, para que los notebooks de análisis apliquen limpieza e imputación.

In [1]:
# ── Celda 1: Importaciones y parámetros base ──
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
N = 1500

In [2]:
# ── Celda 2: Generación de las condiciones a partir de las 4 zonas latentes ──
# Perfiles de las 4 zonas agroecológicas (medias por variable de condición)
# Orden: temp, humedad, ph, sol, riego, fertilizante, densidad
zonas = {
    'Optima':        {'p': 0.35, 'mean': [20.0, 60, 5.5, 9.0, 550, 200, 4100]},
    'EstresHidrico': {'p': 0.25, 'mean': [22.5, 40, 5.6, 9.5, 700, 210, 3800]},
    'DesbalancePH':  {'p': 0.20, 'mean': [20.5, 58, 6.9, 8.8, 560, 265, 4200]},
    'MultiEstres':   {'p': 0.20, 'mean': [22.2, 48, 6.2, 7.4, 600, 235, 3550]},
}
std = np.array([0.80, 4.3, 0.20, 0.52, 42, 30, 250])  # dispersion calibrada

nombres = list(zonas.keys())
probs = [zonas[z]['p'] for z in nombres]
etiqueta = rng.choice(len(nombres), size=N, p=probs)

cond = np.zeros((N, 7))
for i, z in enumerate(nombres):
    mask = etiqueta == i
    cond[mask] = rng.normal(np.array(zonas[z]['mean']), std, size=(mask.sum(), 7))

temp, hum, ph, sol, riego, fert, dens = cond.T

# Recorte a rangos plausibles por variable
temp = np.clip(temp, 14, 26)
hum = np.clip(hum, 20, 95)
ph = np.clip(ph, 4.2, 8.0)
sol = np.clip(sol, 5, 12)
riego = np.clip(riego, 200, 900)
fert = np.clip(fert, 18, 617)
dens = np.clip(dens, 2000, 6000)

In [3]:
# ── Celda 3: Variables objetivo (rendimiento y calidad) ──
# Rendimiento: funcion de respuesta agronomica con penalizacion cuadratica
rend = (9200
        - 70 * (temp - 20.0) ** 2      # optimo de temperatura 20 C
        - 450 * (ph - 5.5) ** 2        # optimo de pH 5.5
        - 3.0 * np.abs(hum - 60)       # optimo de humedad 60 %
        + 60 * (sol - 7)
        + rng.normal(0, 300, N))
rend = np.clip(rend, 3000, 11000).round(0)

# Calidad por terciles del rendimiento
q1, q2 = np.quantile(rend, [0.33, 0.66])
calidad = np.where(rend <= q1, 'Baja', np.where(rend <= q2, 'Media', 'Alta'))

In [4]:
# ── Celda 4: Ensamblado del DataFrame, outliers y nulos ──
df = pd.DataFrame({
    'id_parcela': [f'PARC-{i+1:05d}' for i in range(N)],
    'region': rng.choice(['La Libertad', 'Lambayeque', 'Ica'], N, p=[0.5, 0.3, 0.2]),
    'variedad': rng.choice(['Biloxi', 'Ventura', 'Emerald', 'Rocio'], N),
    'superficie_ha': rng.uniform(2, 12, N).round(2),
    'temp_promedio': temp.round(1),
    'humedad_suelo': hum.round(1),
    'ph_suelo': ph.round(2),
    'horas_sol': sol.round(1),
    'riego_mm': riego.round(0),
    'fertilizante_kg': fert.round(0),
    'densidad_plantas': dens.round(0),
    'rendimiento_kg_ha': rend,
    'calidad_cosecha': calidad,
})

# Valores atipicos en sensores (algunos valores imposibles)
oh = rng.choice(N, 8, replace=False)
df.loc[oh, 'humedad_suelo'] = rng.uniform(105, 125, 8).round(1)
op = rng.choice(N, 6, replace=False)
df.loc[op, 'ph_suelo'] = rng.uniform(9.5, 12, 6).round(2)

# Nulos intencionales en columnas de sensores
for col, frac in [('humedad_suelo', 0.04), ('ph_suelo', 0.03), ('horas_sol', 0.025)]:
    idx = rng.choice(N, int(frac * N), replace=False)
    df.loc[idx, col] = np.nan
    

In [5]:
# ── Celda 5: Exportacion a CSV ──
df.to_csv('andinasense_parcelas.csv', index=False)
print('Dataset exportado: andinasense_parcelas.csv')
print('Dimensiones:', df.shape)

Dataset exportado: andinasense_parcelas.csv
Dimensiones: (1500, 13)


In [6]:
# ── Celda 6: Verificacion basica ──
print('Columnas:', list(df.columns))
print('\nNulos por columna de sensor:')
print(df[['humedad_suelo', 'ph_suelo', 'horas_sol']].isna().sum())
print('\nProporcion de calidad_cosecha:')
print(df['calidad_cosecha'].value_counts(normalize=True).round(3))
print('\nEstadisticas de rendimiento_kg_ha:')
print(df['rendimiento_kg_ha'].describe().round(1))

Columnas: ['id_parcela', 'region', 'variedad', 'superficie_ha', 'temp_promedio', 'humedad_suelo', 'ph_suelo', 'horas_sol', 'riego_mm', 'fertilizante_kg', 'densidad_plantas', 'rendimiento_kg_ha', 'calidad_cosecha']

Nulos por columna de sensor:
humedad_suelo    60
ph_suelo         45
horas_sol        37
dtype: int64

Proporcion de calidad_cosecha:
calidad_cosecha
Alta     0.34
Baja     0.33
Media    0.33
Name: proportion, dtype: float64

Estadisticas de rendimiento_kg_ha:
count     1500.0
mean      8795.9
std        523.6
min       6740.0
25%       8439.0
50%       8832.0
75%       9172.8
max      10131.0
Name: rendimiento_kg_ha, dtype: float64
